# Data aggregation for ML training

In [46]:
import os
import pandas as pd

Load data

In [ ]:
df_matches = pd.read_csv("data/matches_clean.csv")
df_goals = pd.read_csv("data/goals_clean.csv")
df_players = pd.read_csv("data/players_clean.csv")

Create goal stats

In [48]:
team_goals = (
    df_goals
    .groupby("scoring_team")
    .size()
    .reset_index(name="total_goals")
)
team_goals.rename(columns={"scoring_team": "team"}, inplace=True)

team_goals.head()

,team,total_goals
0,Algeria,7
1,Angola,1
2,Argentina,52
3,Australia,17
4,Belgium,29


In [49]:
players_with_goals = (
    df_goals
    .groupby("scoring_team")["player_id"]
    .nunique()
    .reset_index(name="num_players_with_goals")
)
players_with_goals.rename(columns={"scoring_team": "team"}, inplace=True)


players_with_goals.head()

,team,num_players_with_goals
0,Algeria,5
1,Angola,1
2,Argentina,22
3,Australia,10
4,Belgium,16


In [50]:
player_goal_counts = (
    df_goals
    .groupby(["scoring_team", "player_id"])
    .size()
    .reset_index(name="goals_by_player")
)
player_goal_counts.rename(columns={"scoring_team": "team"}, inplace=True)

max_goals_player = (
    player_goal_counts
    .groupby("team")["goals_by_player"]
    .max()
    .reset_index(name="max_goals_by_single_player")
)

max_goals_player.head()

,team,max_goals_by_single_player
0,Algeria,2
1,Angola,1
2,Argentina,13
3,Australia,5
4,Belgium,5


Just for fun, let's output the top players, scoring the most goals

In [51]:
max_goals_player_info = (
    player_goal_counts.loc[
        player_goal_counts.groupby("team")["goals_by_player"].idxmax()
    ][["team", "player_id", "goals_by_player"]]
    .rename(columns={"player_id": "top_scorer", "goals_by_player": "max_goals"})
    .reset_index(drop=True)
)

max_goals_player_info = max_goals_player_info.merge(
    df_players[["player_id", "player_name"]],
    left_on="top_scorer",
    right_on="player_id",
    how="left"
)

max_goals_player_info = max_goals_player_info[["team", "top_scorer", "player_name", "max_goals"]]

max_goals_player_info = max_goals_player_info.sort_values(
    by="max_goals",
    ascending=False
).reset_index(drop=True)

max_goals_player_info.head(10)

,team,top_scorer,player_name,max_goals
0,Germany,P-27787,Miroslav Klose,16
1,Argentina,P-14758,Lionel Messi,13
2,France,P-64077,Kylian Mbappé,12
3,Brazil,P-62722,Nazário Ronaldo,11
4,Spain,P-49097,David Villa,9
5,Portugal,P-70442,Cristiano Ronaldo,8
6,England,P-58924,Harry Kane,8
7,Uruguay,P-30486,Luis Suárez,7
8,Croatia,P-93884,Ivan Perišić,6
9,Ghana,P-38556,Asamoah Gyan,6


In [53]:
matches_played = (
    df_matches
    .melt(
        id_vars=["match_id"],
        value_vars=["team1", "team2"],
        value_name="team"
    )
    .groupby("team")
    .nunique()["match_id"]
    .reset_index(name="total_matches")
)

matches_played.head()

,team,total_matches
0,Algeria,7
1,Angola,3
2,Argentina,31
3,Australia,17
4,Belgium,19


In [55]:
matches_with_goals = (
    df_goals
    .groupby("scoring_team")["match_id"]
    .nunique()
    .reset_index(name="matches_with_goals")
)
matches_with_goals.rename(columns={"scoring_team": "team"}, inplace=True)

matches_with_goals.head()

,team,matches_with_goals
0,Algeria,4
1,Angola,1
2,Argentina,25
3,Australia,12
4,Belgium,14


Merge stats into team features

In [56]:
team_features = (
    team_goals
    .merge(players_with_goals, on="team", how="left")
    .merge(max_goals_player, on="team", how="left")
    .merge(matches_played, left_on="team", right_on="team", how="left")
    .merge(matches_with_goals, left_on="team", right_on="team", how="left")
)

team_features.head()

,team,total_goals,num_players_with_goals,max_goals_by_single_player,total_matches,matches_with_goals
0,Algeria,7,5,2,7,4
1,Angola,1,1,1,3,1
2,Argentina,52,22,13,31,25
3,Australia,17,10,5,17,12
4,Belgium,29,16,5,19,14


In [57]:
team_features["matches_without_goals"] = (
    team_features["total_matches"] - team_features["matches_with_goals"]
)

team_features["avg_goals_per_match"] = (
    team_features["total_goals"]
    / team_features["total_matches"]
)

team_features.head()

,team,total_goals,num_players_with_goals,max_goals_by_single_player,total_matches,matches_with_goals,matches_without_goals,avg_goals_per_match
0,Algeria,7,5,2,7,4,3,1.000000
1,Angola,1,1,1,3,1,2,0.333333
2,Argentina,52,22,13,31,25,6,1.677419
3,Australia,17,10,5,17,12,5,1.000000
4,Belgium,29,16,5,19,14,5,1.526316


Add two rows per match (for team1 & team2) for ML training

In [58]:
team1_rows = df_matches.loc[:, ["match_id", "team1", "team2", "team1_score", "team2_score"]].copy()
team1_rows["win"] = (team1_rows["team1_score"] > team1_rows["team2_score"]).astype(int)

team2_rows = (df_matches[["match_id", "team1", "team2", "team1_score", "team2_score"]]
    .rename(columns={
        "team1": "team2",
        "team2": "team1",
        "team1_score": "team2_score",
        "team2_score": "team1_score",
    }).copy())
team2_rows["win"] = (team2_rows["team1_score"] > team2_rows["team2_score"]).astype(int)

df_matches_labelled = pd.concat([team1_rows, team2_rows], ignore_index=True).sort_values(by="match_id").reset_index(drop=True)
df_matches_labelled.drop(columns=["team1_score", "team2_score"], inplace=True)

df_matches_labelled.head(10)

,match_id,team1,team2,win
0,M-2002-01,France,Senegal,0
1,M-2002-01,Senegal,France,1
2,M-2002-02,Cameroon,Republic of Ireland,0
3,M-2002-02,Republic of Ireland,Cameroon,0
4,M-2002-03,Denmark,Uruguay,1
5,M-2002-03,Uruguay,Denmark,0
6,M-2002-04,Germany,Saudi Arabia,1
7,M-2002-04,Saudi Arabia,Germany,0
8,M-2002-05,Argentina,Nigeria,1
9,M-2002-05,Nigeria,Argentina,0


Join team features to matches

In [59]:
matches_with_features = df_matches_labelled.merge(
    team_features,
    left_on="team1",
    right_on="team",
    how="left",
    suffixes=("", "_team1")
)

matches_with_features = matches_with_features.merge(
    team_features,
    left_on="team2",
    right_on="team",
    how="left",
    suffixes=("_team1", "_team2")
)


matches_with_features["goals_diff"] = matches_with_features["total_goals_team1"] - matches_with_features["total_goals_team2"]
matches_with_features["matches_diff"] = matches_with_features["total_matches_team1"] - matches_with_features["total_matches_team2"]
matches_with_features["avg_goals_diff"] = matches_with_features["avg_goals_per_match_team1"] - matches_with_features["avg_goals_per_match_team2"]
matches_with_features["num_players_with_goals_diff"] = matches_with_features["num_players_with_goals_team1"] - matches_with_features["num_players_with_goals_team2"]
matches_with_features["max_goals_diff"] = matches_with_features["max_goals_by_single_player_team1"] - matches_with_features["max_goals_by_single_player_team2"]

matches_with_features = matches_with_features.drop(columns=["team_team1", "team_team2"], errors="ignore")

matches_with_features.head()

,match_id,team1,team2,win,total_goals_team1,num_players_with_goals_team1,max_goals_by_single_player_team1,total_matches_team1,matches_with_goals_team1,matches_without_goals_team1,...,max_goals_by_single_player_team2,total_matches_team2,matches_with_goals_team2,matches_without_goals_team2,avg_goals_per_match_team2,goals_diff,matches_diff,avg_goals_diff,num_players_with_goals_diff,max_goals_diff
0,M-2002-01,France,Senegal,0,50.0,24.0,12.0,32.0,22.0,10.0,...,3.0,12.0,8.0,4.0,1.333333,34.0,20.0,0.229167,11.0,9.0
1,M-2002-01,Senegal,France,1,16.0,13.0,3.0,12.0,8.0,4.0,...,12.0,32.0,22.0,10.0,1.562500,-34.0,-20.0,-0.229167,-11.0,-9.0
2,M-2002-02,Cameroon,Republic of Ireland,0,9.0,6.0,3.0,12.0,7.0,5.0,...,3.0,4.0,4.0,0.0,1.500000,3.0,8.0,-0.750000,2.0,0.0
3,M-2002-02,Republic of Ireland,Cameroon,0,6.0,4.0,3.0,4.0,4.0,0.0,...,3.0,12.0,7.0,5.0,0.750000,-3.0,-8.0,0.750000,-2.0,0.0
4,M-2002-03,Denmark,Uruguay,1,12.0,7.0,5.0,14.0,9.0,5.0,...,7.0,22.0,16.0,6.0,1.272727,-16.0,-8.0,-0.415584,-5.0,-2.0


Output file

In [ ]:
os.makedirs("output", exist_ok=True)

team_features.to_csv(
    "model/team_stats.csv",
    index=False
)

matches_with_features.to_csv(
    "model/matches_with_features.csv",
    index=False
)